# AI-Powered Study Assistant — Phase 1 Colab Reproducibility Template

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BahuleyaMahesh/MLOps/blob/main/notebooks/colab_reproducibility.ipynb)

This Google Colab notebook provides an end-to-end, one-click environment to clone the repository, install pinned dependencies, run the complete DVC pipeline (`ingest` → `clean` → `explore` → `features` → `train` → `evaluate`), verify all 29 automated tests, and inspect MLflow experiment metrics.

## 1. Environment Setup & Repository Clone

In [ ]:
# Clone the repository if running on Colab
import os
if not os.path.exists('AI-Powered-Study-Assistant'):
    !git clone https://github.com/BahuleyaMahesh/MLOps.git AI-Powered-Study-Assistant
    %cd AI-Powered-Study-Assistant
else:
    %cd AI-Powered-Study-Assistant

!python --version

## 2. Install Pinned Dependencies

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install -r requirements-dev.txt

## 3. Reproduce the Complete DVC Pipeline (`dvc repro`)

Executes all 5 stages in order:
1. `ingest`: Downloads raw EngSAF dataset from Hugging Face.
2. `clean`: Sanitizes HTML, normalizes text, validates score labels.
3. `explore`: Generates dataset profile & EDA report.
4. `features`: Extracts lexical, length, and dense semantic interaction features without target leakage.
5. `train`: Fits M2-A, M2-B, M2-C, M2-D, M2-E models and saves artifacts to `models/`.
6. `evaluate`: Evaluates test split, runs 5-fold GroupKFold cross-validation, and performs ablation studies.

In [ ]:
!dvc repro

## 4. Run Automated Test Suite (29 Tests)

In [ ]:
!pytest -v tests/

## 5. Independent Clean-Clone Reproducibility Verification

Verifies that all metrics reproduce within strict numerical tolerance $\Delta \le 0.001$.

In [ ]:
!python -m src.reproducibility_check

## 6. Inspect Model Comparison and Evaluation Reports

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

with open('reports/baseline_metrics.json') as f:
    metrics = json.load(f)

print('Standard Split Model Results:')
std_models = []
for k, v in metrics['standard_split_models'].items():
    t = v['test']
    std_models.append({
        'Model ID': k,
        'Macro-F1': round(t['macro_f1'], 4),
        'QWK': round(t['quadratic_weighted_kappa'], 4),
        'Accuracy': round(t['accuracy'], 4),
        'MAE': round(t['mae'], 4),
    })
display(pd.DataFrame(std_models))

print('\nQuestion-Grouped 5-Fold GroupKFold CV Results (Zero Question Leakage):')
gkf_models = []
for k, v in metrics['question_grouped_kfold_cv'].items():
    gkf_models.append({
        'Model ID': k,
        'Macro-F1 Mean': round(v['macro_f1_mean'], 4),
        'QWK Mean': round(v['qwk_mean'], 4),
        'Accuracy Mean': round(v['accuracy_mean'], 4),
        'MAE Mean': round(v['mae_mean'], 4),
    })
display(pd.DataFrame(gkf_models))

display(Markdown(open('reports/model2_comparison.md').read()))